# Computer Shop Sales — Exploratory Data Analysis**Objective:** Understand revenue drivers, brand/category performance, salespersonproductivity, and payment behavior in the Computer Shop dataset ahead of Power BI /Streamlit builds.**Tools:** pandas, matplotlib, seaborn

## 1. Setup & Load

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snssns.set_theme(style="whitegrid", palette="deep")plt.rcParams["figure.figsize"] = (10, 5)plt.rcParams["axes.titlesize"] = 13pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_excel("computer shop.xlsx")print(f"Shape: {df.shape}")df.head()

## 2. Structure & Quality Audit

In [ ]:
df.info()

In [ ]:
missing = df.isnull().sum()missing_pct = (missing / len(df) * 100).round(2)quality = pd.DataFrame({"missing": missing, "missing_%": missing_pct, "dtype": df.dtypes})quality[quality["missing"] > 0].sort_values("missing", ascending=False)

In [ ]:
print("Duplicate rows:", df.duplicated().sum())df = df.drop_duplicates()df["Date"] = pd.to_datetime(df["Date"], errors="coerce")if "Total Price" not in df.columns or df["Total Price"].isnull().all():    df["Total Price"] = df["Quantity"] * df["Unit Price"]df.describe(include="all").T

> **Insight cell:** Note row/column count, date range, and which columns had meaningful> missingness (e.g. Salesperson or Payment Method gaps often signal manual entry issues).

## 3. Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))sns.histplot(df["Total Price"], bins=40, kde=True, ax=axes[0]).set_title("Sale Value Distribution")sns.histplot(df["Unit Price"], bins=40, kde=True, ax=axes[1]).set_title("Unit Price Distribution")sns.histplot(df["Quantity"], bins=20, kde=True, ax=axes[2]).set_title("Quantity per Invoice")plt.tight_layout()plt.show()

In [ ]:
plt.figure(figsize=(10, 5))sns.countplot(data=df, y="Category", order=df["Category"].value_counts().index)plt.title("Invoice Volume by Product Category")plt.show()

In [ ]:
plt.figure(figsize=(10, 5))sns.countplot(data=df, y="Brand", order=df["Brand"].value_counts().index)plt.title("Invoice Volume by Brand")plt.show()

## 4. Bivariate & Correlation Analysis

In [ ]:
numeric_cols = ["Quantity", "Unit Price", "Total Price"]corr = df[numeric_cols].corr()plt.figure(figsize=(6, 5))sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")plt.title("Correlation Matrix")plt.show()

In [ ]:
cat_rev = df.groupby("Category")["Total Price"].sum().sort_values(ascending=False)plt.figure(figsize=(10, 5))sns.barplot(x=cat_rev.values, y=cat_rev.index, color="#1E2761")plt.title("Total Revenue by Category")plt.xlabel("Revenue")plt.show()

In [ ]:
brand_rev = df.groupby("Brand")["Total Price"].sum().sort_values(ascending=False)plt.figure(figsize=(10, 5))sns.barplot(x=brand_rev.values, y=brand_rev.index, color="#F2B134")plt.title("Total Revenue by Brand")plt.xlabel("Revenue")plt.show()

In [ ]:
plt.figure(figsize=(9, 5))sns.countplot(data=df, x="Payment Method", order=df["Payment Method"].value_counts().index)plt.title("Transactions by Payment Method")plt.show()

> **Insight cell:** Call out which category/brand combination drives the most revenue vs.> the most transactions — high transaction count with low revenue often signals a> low-ticket accessory category, worth noting separately from headline revenue drivers.

## 5. Time-Series Analysis

In [ ]:
monthly = df.set_index("Date").resample("ME")["Total Price"].sum()fig, ax = plt.subplots(figsize=(12, 5))monthly.plot(ax=ax, marker="o", color="#1E2761")ax.set_title("Monthly Revenue Trend")ax.set_ylabel("Revenue")plt.show()

In [ ]:
df["Month Name"] = df["Date"].dt.month_name()month_order = ["January","February","March","April","May","June",               "July","August","September","October","November","December"]seasonal = df.groupby("Month Name")["Total Price"].sum().reindex(month_order)plt.figure(figsize=(11, 5))sns.barplot(x=seasonal.index, y=seasonal.values, color="#4C72B0")plt.title("Seasonality — Total Revenue by Calendar Month")plt.xticks(rotation=45)plt.show()

## 6. Salesperson & Branch Analysis

In [ ]:
top_sales = df.groupby("Salesperson")["Total Price"].sum().sort_values(ascending=False).head(10)plt.figure(figsize=(9, 5))sns.barplot(x=top_sales.values, y=top_sales.index, color="#55A868")plt.title("Top 10 Salespeople by Revenue")plt.show()

In [ ]:
branch_perf = df.groupby("Branch")["Total Price"].agg(["sum", "count"]).rename(    columns={"sum": "Revenue", "count": "Invoices"}).sort_values("Revenue", ascending=False)branch_perf["Avg Sale Value"] = (branch_perf["Revenue"] / branch_perf["Invoices"]).round(2)branch_perf

## 7. Summary of Key Findings

Replace with your actual findings once you've run the notebook, e.g.:- Highest-revenue category vs. highest-transaction-volume category (often different)- Top brand by revenue and whether it matches top brand by units sold- Peak sales months / seasonality pattern- Top salesperson's contribution as % of total revenue- Dominant payment method and whether it varies by branch